# 第13回　学習曲線とハイパーパラメータ
***
> **前提**: 第6回の交差検証を発展させ，過学習の判断方法を学びます。

## 目次
1. learning_curve
2. validation_curve
3. 交差検証の復習
4. 過学習の判断

---

## この回で学ぶこと

### バイアスとバリアンスのトレードオフ

機械学習モデルの「誤差」は2種類に分解できる：

```
汎化誤差 = バイアス² + バリアンス + ノイズ

【高バイアス（Underfitting）】
  モデルが単純すぎ → 訓練データにすら当てはまらない
  → 訓練スコア低，テストスコア低
  → 解決策：モデルを複雑にする（max_depth を増やすなど）

【高バリアンス（Overfitting）】
  モデルが複雑すぎ → 訓練データは完璧だが，テストデータは失敗
  → 訓練スコア高，テストスコア低（大きく差がある）
  → 解決策：正則化，データを増やす，モデルを単純にする
```

### 学習曲線（Learning Curve）

**訓練データのサイズを変えながら，訓練スコアとテストスコアの変化をプロットしたもの**。

```
【良い学習曲線のパターン】
訓練スコア  ↓  ̄ ̄ ̄\        ← データが増えると訓練スコアは少し下がる
検証スコア  /          ← データが増えると検証スコアは上がる
           両者が収束 → underfittingもoverfittingもない

【過学習の場合】
訓練スコア  ̄ ̄ ̄ ̄ ̄ ̄（高いまま）
検証スコア  ___________（低いまま，差が大きい）
```

- データが少ない時に大きな差がある：正則化またはデータ収集が必要
- データが多くなっても差が縮まらない：モデルが複雑すぎ

### 検証曲線（Validation Curve）

**ハイパーパラメータを変化させながら，訓練スコアと検証スコアをプロット**。最適なハイパーパラメータを視覚的に選ぶための手法だ。

```
【決定木の max_depth の場合】
max_depth が小さい：高バイアス（underfitting）
max_depth が大きい：高バリアンス（overfitting）
最適 max_depth：検証スコアが最大になる点
```

### 交差検証（Cross-Validation）の重要性

単一の train/test 分割では，その分割方法に結果が依存してしまう（運次第）。**k-fold 交差検証** はデータをk個に分割し，各分割でテストデータを入れ替えてk回評価することで，より安定した評価ができる。

> **卒業研究での指針**: 論文でハイパーパラメータを報告する際は，必ず「どのデータで選んだか（交差検証 or テストデータ）」を明記すること。テストデータでハイパーパラメータを選ぶと**テストデータのリーク**になる。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import learning_curve, validation_curve
from sklearn.tree import DecisionTreeClassifier


## 問題1　学習曲線の描画
***

### `cv=5` の意味

`cv=5` は5分割交差検証を意味する。各訓練サイズで：
1. データを5つに分割
2. 4つを訓練，1つを検証に使って評価
3. これを5回繰り返し，5つのスコアの平均と標準偏差を算出

`train_scores` の shape は `(train_sizes の個数, cv=5)` になる。そのため `np.mean(train_scores, axis=1)` で各サイズの平均を取る。

### 標準偏差の帯を描く方法

各点での不確かさ（ばらつき）を帯として描くと，グラフの信頼性が上がる：

```python
plt.fill_between(
    train_sizes,
    train_mean - train_std,
    train_mean + train_std,
    alpha=0.2
)
```

論文やレポートではこのような信頼区間を付けることが推奨される。

### 課題

乳がんデータ `load_breast_cancer()` を用い，`DecisionTreeClassifier(random_state=0)` の **learning_curve** を計算してください（`cv=5`, `train_sizes=np.linspace(0.1, 1.0, 10)`）。

訓練スコアと交差検証スコアの平均を，訓練データサイズに対する折れ線グラフで描画してください。凡例（訓練スコア / 検証スコア）も付けてください。

> **観察ポイント**: 訓練スコアと検証スコアの差が大きい場合，過学習が起きている。データを増やすほど差は縮まるか？

#### Hints
- `learning_curve` の返り値は `(train_sizes, train_scores, val_scores)` の3つ
- `train_scores` は `(訓練サイズ数, cv数)` の2次元配列。各サイズでの平均スコアを求めるには `axis=1` 方向に平均を取る
- グラフには訓練スコアと検証スコアの両方を異なる色でプロットし、凡例を付けること

In [ ]:
# learning_curve
# ここにあなたのコードを書いてください


## 問題2　検証曲線によるハイパーパラメータ探索
***

### `max_depth=None` の意味

`max_depth=None` は木の深さを制限しない（完全に成長させる）ことを意味する。決定木は訓練データを完璧に記憶できるため，`max_depth=None` では訓練スコア ≈ 1.0 になるが，テストデータでは大きく精度が落ちる（過学習の極端な例）。

### グラフの横軸について

`max_depth=[1, 2, 3, 5, 10, 20, None]` の `None` は数値ではないため，グラフ描画には工夫が必要だ。文字列ラベルとして扱うのが一般的だ：

```python
param_range = [1, 2, 3, 5, 10, 20, None]
x_labels = [str(p) for p in param_range]  # ["1", "2", "3", "5", "10", "20", "None"]
plt.xticks(range(len(x_labels)), x_labels)
```

### このグラフから何を読み取るか

検証曲線の典型的なパターン：
- `max_depth` が小さい（1〜2）: 訓練も検証も低い → 高バイアス（underfitting）
- `max_depth` が中程度（3〜5程度）: 検証スコアが最大 → 最適な複雑さ
- `max_depth` が大きい（10以上, None）: 訓練スコアは高いが検証スコアは低い → 高バリアンス（overfitting）

### 課題

同データで `max_depth` を `[1, 2, 3, 5, 10, 20, None]` と変化させ，`validation_curve` を用いて交差検証スコアを取得してください。

横軸: max_depth，縦軸: スコア の折れ線グラフを描画し（訓練と検証の両方），**過学習が始まる max_depth の値**を確認してください。

#### Hints
- `validation_curve` の引数: モデルオブジェクト, X, y, `param_name`（文字列）, `param_range`（値のリスト）, `cv`
- 返り値は `(train_scores, val_scores)` の2つ。どちらも `(param_range の長さ, cv数)` の2次元配列
- `param_range` に `None` が含まれるため、横軸はそのままでは数値として扱えない。文字列ラベルとして扱うと描画しやすい

In [ ]:
# validation_curve
# ここにあなたのコードを書いてください


## 問題3　過学習の診断と対処
***

### 「訓練スコアと検証スコアの差が大きい」の意味

この状態を診断するために，問題1・2の学習曲線と検証曲線を見返そう：

- **差が大きい + 両者が収束しない**: 高バリアンス（過学習）。モデルが訓練データを「暗記」している状態
- **差が小さいが両者とも低い**: 高バイアス（過少適合）。モデルが単純すぎてパターンを学習できていない

対処法のまとめ：

| 問題 | 症状 | 対処法 |
|---|---|---|
| 過学習（高バリアンス） | 訓練高・テスト低 | 正則化，max_depth 制限，データ追加 |
| 過少適合（高バイアス） | 訓練低・テスト低 | モデルを複雑にする，特徴量を増やす |

### max_depth=3 を選ぶ根拠

問題2の検証曲線で max_depth=3 あたりが検証スコアの最大値になるはずだ。このように「検証データ（またはクロスバリデーション）でハイパーパラメータを選ぶ」プロセスが正しい方法だ。

### 課題

learning_curve の結果から，「訓練スコアと検証スコアの差が大きい」状態が何を意味するか，**print または コメント**で2〜3文で説明してください。

そのうえで，`max_depth=3` の決定木のテスト正解率を `train_test_split(test_size=0.2, random_state=0)` で求めてください。


In [ ]:
# 過学習の説明と max_depth=3 の評価
# ここにあなたのコードを書いてください


## 問題4　過学習の数値的確認と次のステップ
***

### 訓練精度とテスト精度の差で過学習を定量化

グラフで定性的に確認するだけでなく，数値で差を計算することが重要だ：

```python
gap = train_acc - test_acc
# gap が 0.05（5ポイント）以上なら過学習が疑われる
```

### GridSearchCV で自動的にハイパーパラメータを探索する方法（発展）

本回では手動で max_depth を変化させたが，実際の研究では `GridSearchCV` で自動化する：

```python
from sklearn.model_selection import GridSearchCV

param_grid = {"max_depth": [1, 2, 3, 5, 10, 20, None]}
gs = GridSearchCV(DecisionTreeClassifier(random_state=0), param_grid, cv=5)
gs.fit(X_train, y_train)
print(f"最適 max_depth: {gs.best_params_}")
print(f"最高CV スコア: {gs.best_score_:.4f}")
```

`GridSearchCV` は全組み合わせを試すが，パラメータが多い場合は `RandomizedSearchCV`（ランダムに組み合わせを選ぶ）の方が効率的だ。

### 課題

`max_depth=3` と `max_depth=20` の2モデルについて，訓練データとテストデータそれぞれの正解率を比較する表（DataFrame）を作成して出力してください。

それぞれのモデルで「訓練精度 - テスト精度」の差（gap）も計算し，どちらが過学習しているかを数値で示してください。

> **考えてみよう**: max_depth=20 のモデルは訓練データを「暗記」しているため，訓練精度が非常に高い。しかしテストデータでは…？


In [ ]:
# 2モデルの train/test 正解率比較
# ここにあなたのコードを書いてください
